## Working with Data

Pytorch has two **Primitive to work with data**:  *torch.utils.data.DataLoader* and *torch.utils.data.Dataset* .
1. *Dataset* :- stores the samples and their corresponding labels
2. *DataLoader* :- Wraps a iterable around the dataset.



In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

Pytorch offers domain-specific libraries such as **TorchText**, **TorchVision** and **TorchAudio** , all of which include datasets . For this tutorial we will be using **TorchVision Dataset**.

The **TorchVision Dataset** module contains *Dataset* Object for many of real-world vision data like **CIFAR**, **COCO** . For this tutorial we will use **FashionMNIST** Dataset. Every **TorchVision Dataset** includes two arguments : **Transform** and **target_transform** to modify the samples and labels respectively.

In [3]:
# Download training data from Open Datasets.
training_data = datasets.FashionMNIST(
    root= "data",
    train= True,
    download = True,
    transform = ToTensor(),
)

# Download test data from open Datasets.
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor()
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 15.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 302kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 4.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.5MB/s]


We pass the **Dataset** as an argument to **DataLoader**. This wraps an iterable over our dataset, and supports automatic batching, sampling, shuffling and multiprocess data loading. Here we define a batch size of 64, i.e. each element in the dataloader iterable will return a batch of 64 features and labels.

In [4]:
batch_size = 64

#Create data loaders
train_data_loaders = DataLoader(training_data, batch_size=batch_size)
test_data_loaders = DataLoader(test_data,batch_size=batch_size)


for X, y in test_data_loaders:
  print(f" Shape of X [N, C, H , W]:{X.shape}")
  print(f"Shape of y :{y.shape} {y.dtype}")
  break


 Shape of X [N, C, H , W]:torch.Size([64, 1, 28, 28])
Shape of y :torch.Size([64]) torch.int64


## Creating Model

To define a neural network in PyTorch, we create a class that inherits from **nn.Module**. We define the layers of the network in the **__init__** function and specify how data will pass through the network in the **forward** function. To accelerate operations in the neural network, we move it to the **accelerator** such as CUDA, MPS, MTIA, or XPU. If the current accelerator is available, we will use it. Otherwise, we use the CPU.

In [14]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Optimizing the Model Parameters:
To train a model , we need a loss function and an optmizer:

In [7]:
loss_fn = nn.CrossEntropyLoss()
optmizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training dataset (fed to it in batches), and backpropagates the prediction error to adjust the model’s parameters.

In [11]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model’s performance against the test dataset to ensure it is learning.



In [12]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [15]:
epochs = 5
for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  train(train_data_loaders, model, loss_fn, optmizer)
  test(test_data_loaders, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.316257  [   64/60000]
loss: 2.317178  [ 6464/60000]
loss: 2.314547  [12864/60000]
loss: 2.303764  [19264/60000]
loss: 2.306499  [25664/60000]
loss: 2.308822  [32064/60000]
loss: 2.304260  [38464/60000]
loss: 2.311551  [44864/60000]
loss: 2.309572  [51264/60000]
loss: 2.294443  [57664/60000]
Test Error: 
 Accuracy: 10.3%, Avg loss: 2.309329 

Epoch 2
-------------------------------
loss: 2.316257  [   64/60000]
loss: 2.317178  [ 6464/60000]
loss: 2.314547  [12864/60000]
loss: 2.303764  [19264/60000]
loss: 2.306499  [25664/60000]
loss: 2.308822  [32064/60000]
loss: 2.304260  [38464/60000]
loss: 2.311551  [44864/60000]
loss: 2.309572  [51264/60000]
loss: 2.294443  [57664/60000]
Test Error: 
 Accuracy: 10.3%, Avg loss: 2.309329 

Epoch 3
-------------------------------
loss: 2.316257  [   64/60000]
loss: 2.317178  [ 6464/60000]
loss: 2.314547  [12864/60000]
loss: 2.303764  [19264/60000]
loss: 2.306499  [25664/60000]
loss: 2.308822  [32064/600

## Saving Model

A common way to save a model is to serialize the internal state dictionary (containing the model parameters).

In [16]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


### Loading Models

The process for loading a model includes re-creating the model structure and loading the state dictionary into it.

In [17]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [18]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Sneaker", Actual: "Ankle boot"
